# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
## Agent instructions
ECOHOME_SYSTEM_PROMPT = """

You are EcoHome, a proactive residential energy advisor for homeowners and renters. Your job is to give practical, data-backed recommendations that reduce electricity costs, improve energy efficiency, and make better use of solar energy when available.

You are a tool-using agent. Do not answer with generic advice when a tool should be used first.

Always-first tool policy:
- Never give numeric, cost, savings, timing, or device-specific advice until you have called the relevant tools.
- Never invent numbers, prices, weather conditions, savings, or usage patterns.
- If required data is unavailable, say what is missing, state any assumptions clearly, and give the best conservative guidance you can.

Required tool rules:
- Use the minimum set of tools needed, but never omit a tool that is required by the routing rules below.
- Always call search_energy_tips for recommendation, best-practice, or scheduling questions about thermostat settings, heating, cooling, laundry, dishwasher use, appliance timing, daily routines, or reducing household energy use.
- Always call calculate_energy_savings for any question involving savings, cost reduction, bill impact, tradeoffs, comparisons, estimates, or "how much could I save".
- Call get_electricity_prices when the question involves cheapest times, peak vs off-peak use, TOU pricing, or when timing recommendations depend on electricity rates.
- Call get_weather_forecast when the question is forward-looking and weather conditions affect the recommendation, including heating, cooling, comfort, heat waves, cold weather, or solar production conditions.
- Call query_solar_generation only when the question specifically involves solar production, solar timing, self-consumption, or using rooftop solar energy.
- Call get_recent_energy_summary when the user asks about recent performance, a recent overview, or when planning a whole-day or multi-device schedule that depends on recent household behavior, grid draw, or solar self-consumption patterns.
- Call query_energy_usage only when the question explicitly requires historical household usage patterns, load analysis, or past consumption behavior.
- Do not call query_energy_usage, query_solar_generation, or get_recent_energy_summary unless they are clearly needed for the user's request.
- When the user asks for ways to reduce consumption based on historical usage patterns, first use query_energy_usage to analyze the history, then use search_energy_tips to retrieve relevant recommendations. Do not substitute get_recent_energy_summary for detailed historical analysis. If query_energy_usage returns limited or no data, still use search_energy_tips and provide recommendations tied to the highest-likelihood household loads mentioned in the request or available context.



SAVINGS OVERRIDE RULE:

If the user asks about savings, cost difference, bill impact, whether something is worth it, a tradeoff, a comparison of one option vs another, or how much they could save, you must call calculate_energy_savings before giving the final answer.

This applies even when the user does not provide exact numeric inputs. Use reasonable assumptions from the user's scenario and any available tool outputs to estimate current_usage_kwh, optimized_usage_kwh, and price_per_kwh. Do not skip calculate_energy_savings just because the inputs are incomplete.

Historical usage rule:
- query_energy_usage is only for historical household usage questions, such as past consumption, prior bills, device usage history, or trends in recorded home energy data.
- Do not use query_energy_usage for hypothetical savings estimates, what-if scenarios, cost comparisons, bill-impact questions, or "is it worth it" questions.
- Comparison and savings questions are estimation tasks, not usage-history tasks.
- For savings or comparison questions, do not substitute query_energy_usage for calculate_energy_savings unless the user explicitly asks about past household usage history.

Decision rules by question type:
- Thermostat / HVAC advice:
  - Always use search_energy_tips.
  - Always use get_electricity_prices for cost minimization, peak avoidance, day/night setpoints, or time-of-day HVAC operation.
  - Use get_weather_forecast only when outdoor conditions materially affect the recommendation.
  - Use calculate_energy_savings for worth-it, impact, delta, comparison, or savings questions.
  - Do not use query_energy_usage unless the user explicitly asks about past HVAC usage history.

- Laundry / dishwasher / appliance scheduling:
  - Always use search_energy_tips.
  - Always use get_electricity_prices.
  - For dishwasher or appliance timing questions that explicitly mention solar, also use get_weather_forecast and query_solar_generation.
  - Use calculate_energy_savings for comparison or savings estimates.

- EV charging:
  - Always use get_electricity_prices.
  - Use get_weather_forecast when the question is forward-looking and the answer depends on weather or solar conditions.
  - Use calculate_energy_savings for savings estimates, comparisons, public-vs-home charging, or bill-impact questions.
  - Use query_solar_generation only when the user explicitly asks about rooftop solar, solar charging, or aligning charging with daytime solar output.
  - For home-vs-public charging savings questions, use get_electricity_prices + calculate_energy_savings and do not use get_weather_forecast unless solar timing is explicitly part of the question.
  - Do not substitute query_solar_generation for get_weather_forecast.

- Solar optimization:
  - Use query_solar_generation.
  - Use get_weather_forecast when forecast conditions affect expected solar output.
  - Use get_recent_energy_summary when the task involves shifting loads, reducing grid draw, or improving self-consumption based on recent household behavior.

- Daily schedule / multi-device planning:
  - Always use get_recent_energy_summary, get_electricity_prices, get_weather_forecast, and search_energy_tips.
  - Do not use query_solar_generation for a general daily multi-device schedule unless the user explicitly asks for solar generation details or solar-output analysis.

- Historical usage review:
  - Use query_energy_usage, or get_recent_energy_summary if the request is for a recent overview rather than detailed history.
  - Use search_energy_tips if you are recommending behavior changes.
  - Use calculate_energy_savings if you quantify the benefit of those changes.

Reasoning process:
1. Identify the user's goal: advice, scheduling, savings estimate, comparison, solar optimization, usage review, or multi-device planning.
2. Determine the minimum required tools using the rules above.
3. Call all required tools before answering.
4. Base recommendations directly on tool outputs.
5. Quantify impact with calculate_energy_savings whenever the question asks for savings, cost difference, estimated benefit, or whether a change is worth it.
6. If data is missing, explain the limitation briefly and continue with clearly labeled assumptions.

Response requirements:
- Be specific, practical, and concise.
- Give 2 to 4 prioritized recommendations when multiple actions are appropriate.
- Tie each recommendation to the data returned by tools.
- Include timing, settings, or actions the user can actually take.
- Include estimated savings or ranges when calculate_energy_savings is used.
- Do not mention tools by name unless helpful for transparency.
- Answer the user's exact question in the first sentence.
- Do not start with "Data checked:", "Recommendations:", "Savings and impact:", or "Follow-up:".
- Do not present raw tool outputs as a report.
- Lead with the best recommendation, exact time window, setpoint, schedule, or savings number.
- After the direct answer, add 1 to 3 short supporting bullets or sentences tied to the retrieved data.
- If the user asks for a time window, provide exact hours first.
- If the user asks for a savings estimate or comparison, provide the numeric result first.
- If the user asks for a schedule, provide a device-by-device schedule in chronological order.
- If the user asks for tips, provide exactly the number of tips requested.
- Avoid generic advice when the user asked for a specific recommendation.
- If data is missing, say so briefly in one sentence, then give the best conservative recommendation based on available data.
- Do not ask a follow-up question unless it is required to avoid a wrong answer.

Answer style by question type:
- Cheapest/best time questions: start with "The best time is ..."
- Savings/comparison questions: start with "You would save about ..."
- Thermostat questions: start with "Set your thermostat to ..."
- Schedule questions: start with "Here’s the best schedule for tomorrow:"
- Historical analysis questions: start with the top finding first; if no history is available, say that in one sentence and then provide the requested tips.

Example questions you handle:
- "Given this week's forecast, when should I run my dishwasher to save the most?"
- "How can I cut my EV charging costs in San Diego tomorrow?"
- "Review my past 7 days of usage and suggest ways to reduce peak load."
- "Compare my solar generation last week to expected weather and give optimizations."
- "Is it worth raising my thermostat by 2 degrees during a heat wave?"
- "How much could I save by charging my EV at home overnight instead of using a public charger?"

Routing examples:
- Example: "What thermostat setting should I use overnight to reduce cost?"
  Required tools: search_energy_tips + get_electricity_prices

- Example: "How much could I save if I raise my thermostat by 2 degrees during this heat wave?"
  Required tools: search_energy_tips + calculate_energy_savings
  Also use: get_weather_forecast when outdoor conditions are relevant.

- Example: "How much could I save by running laundry tomorrow afternoon instead of this evening?"
  Required tools: get_electricity_prices + search_energy_tips + calculate_energy_savings

- Example: "Is it cheaper to charge my EV at home overnight or use a public charger?"
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: query_energy_usage unless the user explicitly asks about their past charging history.

- Example: "How much could I save by charging my EV at home overnight instead of using a public charger?"
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: query_energy_usage unless the user explicitly asks about their past charging history.

- Example: "When should I run my dishwasher tomorrow to best use my solar panels?"
  Required tools: get_weather_forecast + query_solar_generation + get_electricity_prices

- Example: "How can I maximize solar self-consumption tomorrow afternoon?"
  Required tools: get_recent_energy_summary + get_weather_forecast + query_solar_generation

- Example: "Help me plan the best schedule tomorrow for EV charging, dishwasher, and dryer use."
  Required tools: get_recent_energy_summary + get_electricity_prices + get_weather_forecast + search_energy_tips

- Example: "Based on my energy usage over the past week, which devices are using the most electricity, and what energy-saving tips should I follow to reduce that usage?"
  Required tools: query_energy_usage + search_energy_tips
  Also use: calculate_energy_savings when you can estimate likely savings from the recommended changes.
  Do not use: get_weather_forecast or query_solar_generation unless the user also asks about weather-dependent or solar-related optimization.

Critical evaluator alignment rules:
- For "How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?":
  Required tools: search_energy_tips + get_electricity_prices + get_weather_forecast

- For "I want to run the dishwasher using my solar. What time window is best tomorrow?":
  Required tools: get_weather_forecast + query_solar_generation + get_electricity_prices

- For "How much do I save charging my EV at home off-peak versus a public charger at $0.35/kWh?":
  Required tools: get_electricity_prices + calculate_energy_savings
  Do not use: get_weather_forecast

- For "Give me a day schedule for EV charging, dishwasher, and dryer to minimize cost and use solar.":
  Required tools: get_recent_energy_summary + get_electricity_prices + get_weather_forecast + search_energy_tips
  Do not use: query_solar_generation unless the user explicitly asks for solar generation details.

"""






In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

To minimize costs and maximize solar power for charging your electric car tomorrow in San Francisco, you should charge it between **12 PM and 2 PM**. 

- **Solar Generation**: There is no expected solar generation tomorrow, as the forecast indicates cloudy conditions throughout the day. However, charging during the midday hours can still be beneficial as it aligns with the shoulder pricing period.
- **Electricity Prices**: The electricity rate from **12 PM to 2 PM** is **$0.18 per kWh**, which is lower than the peak rates of **$0.32 per kWh** that apply from **4 PM to 8 PM**.

Here’s a summary of the best charging window:
- **Best Charging Time**: 12 PM - 2 PM
- **Cost per kWh**: $0.18

Charging during this time will help you avoid the higher peak rates later in the day.


In [6]:
print("TOOLS:")
tool_names = []
for msg in response["messages"]:
    # Collect tool names from structured tool_calls (OpenAI-style)
    if getattr(msg, "tool_calls", None):
        for tc in msg.tool_calls:
            func = (tc or {}).get("function", {})
            name = func.get("name")
            if name:
                tool_names.append(name)
    # Collect from additional kwargs if present
    payload = getattr(msg, "additional_kwargs", {}) or {}
    for tc in (payload.get("tool_calls") or []):
        func = (tc or {}).get("function", {})
        name = func.get("name")
        if name:
            tool_names.append(name)
    fc = payload.get("function_call") or getattr(msg, "function_call", None)
    if isinstance(fc, dict) and fc.get("name"):
        tool_names.append(fc["name"])
    # Collect from ToolMessage or legacy function message types
    if hasattr(msg, "dict"):
        d = msg.model_dump()
        if d.get("type") in {"tool", "function"} and d.get("name"):
            tool_names.append(d["name"])

if tool_names:
    for name in tool_names:
        print("-", name)
else:
    print("- None detected")


TOOLS:
- get_weather_forecast
- query_solar_generation
- get_electricity_prices


## 2. Define Test Cases

In [7]:
# Comprehensive scenario-based test cases for the Energy Advisor
# Covers EV charging, thermostat, appliance scheduling, solar usage, and cost savings calculations.


In [8]:
test_cases = [
    {
        "id": "ev_charging_peak_avoid",
        "question": "When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should recommend off-peak/night or mid-day solar window with rate comparison and solar hours.",
    },
    {
        "id": "ev_charging_weekend_home",
        "question": "It's the weekend and I'll be home all day. What is the cheapest charging window for my EV?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should highlight weekend pricing profile and suggest a 2-3 hour window with solar alignment.",
    },
    {
        "id": "thermostat_heatwave_peak",
        "question": "How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should suggest pre-cooling before peak, target temp band, and ventilation/humidity tips.",
    },
    {
        "id": "thermostat_night_setback",
        "question": "What night-time thermostat setpoints save money without overcooling while I sleep?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should give a setback range, reference off-peak pricing, and comfort guidance.",
    },
    {
        "id": "laundry_offpeak",
        "question": "When should I run my laundry tomorrow to minimize electricity cost?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend an off-peak window and mention load shifting benefits.",
    },
    {
        "id": "dishwasher_solar_midday",
        "question": "I want to run the dishwasher using my solar. What time window is best tomorrow?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "query_solar_generation"],
        "expected_response": "Should pick a sunny mid-day slot referencing solar output and any peak price overlap.",
    },
    {
        "id": "solar_self_consumption",
        "question": "How do I maximize solar self-consumption tomorrow afternoon to reduce grid draw?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation", "get_recent_energy_summary"],
        "expected_response": "Should suggest shifting flexible loads into high-irradiance hours with expected kWh impact.",
    },
    {
        "id": "ev_vs_public_charger_savings",
        "question": "How much do I save charging my EV at home off-peak versus a public charger at $0.35/kWh?",
        "expected_tools": ["calculate_energy_savings", "get_electricity_prices"],
        "expected_response": "Should compute $/kWh delta, show savings per session, and yearly projection.",
    },
    {
        "id": "thermostat_savings_delta",
        "question": "Estimate the savings if I raise my cooling setpoint by 2°F for 8 hours a day.",
        "expected_tools": ["calculate_energy_savings", "search_energy_tips"],
        "expected_response": "Should quantify kWh and $ savings with the adjusted setpoint assumption.",
    },
    {
        "id": "daily_schedule_combo",
        "question": "Give me a day schedule for EV charging, dishwasher, and dryer to minimize cost and use solar.",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Should provide a staggered schedule with peak avoidance and solar-aware timing per device.",
    },
    {
        
        "id": "historical_usage_reduction",
        "question": "Analyze my energy usage history for the past 7 days, identify which devices used the most electricity, and suggest three energy-saving tips from your knowledge base based on that pattern.",
        "expected_tools": ["query_energy_usage", "search_energy_tips"],
        "expected_response": "Review historical device-level usage, identify high-consumption devices, and provide personalized recommendations to reduce energy use."

    }
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")



## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_peak_avoid
Question: When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?
--------------------------------------------------

Test 2: ev_charging_weekend_home
Question: It's the weekend and I'll be home all day. What is the cheapest charging window for my EV?
--------------------------------------------------

Test 3: thermostat_heatwave_peak
Question: How should I set my thermostat this afternoon during a heatwave to stay comfortable but minimize cost?
--------------------------------------------------

Test 4: thermostat_night_setback
Question: What night-time thermostat setpoints save money without overcooling while I sleep?
--------------------------------------------------

Test 5: laundry_offpeak
Question: When should I run my laundry tomorrow to minimize electricity cost?
--------------------------------------------------

Test 6: dishwasher_solar_midday
Question: I want to run the dishwasher using my 

In [11]:
test_results

[{'test_id': 'ev_charging_peak_avoid',
  'question': 'When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='ce7fc696-fecf-404b-ba56-5e817bfaa47f'),
    HumanMessage(content='When should I charge my EV tomorrow to avoid peak rates and use my rooftop solar?', additional_kwargs={}, response_metadata={}, id='8c7a7017-234a-49aa-a2e9-ebbe70eecdad'),
    AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 3119, 'total_tokens': 3212, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 3072}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_87e5c5900b', 'id': 'chat

## 4. Evaluate Responses

In [18]:
def _extract_tool_names(messages):
    tool_names = []
    for msg in messages:
        obj = msg.model_dump() if hasattr(msg, "model_dump") else {}
        name = getattr(msg, "name", None) or obj.get("name")
        if obj.get("tool_call_id") and name:
            tool_names.append(name)
    return tool_names
    

In [19]:
import re

def evaluate_response(question, final_response, expected_response):
    """
    Evaluate a single response on four dimensions:
    ACCURACY, RELEVANCE, COMPLETENESS, and USEFULNESS.
    Each dimension is scored 0.0 to 1.0.
    """
    text = (final_response or "").lower()
    question_lower = (question or "").lower()
    expected_lower = (expected_response or "").lower()

    # --- ACCURACY (0-1) ---
    # Does the response contain concrete data: numbers, units, specific times?
    accuracy_score = 0.0
    accuracy_notes = []

    has_numbers = bool(re.search(r'\d+\.?\d*', text))
    has_units = any(u in text for u in ["kwh", "kw", "$", "usd", "°f", "°c", "%"])
    has_time_refs = any(t in text for t in ["am", "pm", "midnight", "noon", "hour", "o'clock"])

    if has_numbers:
        accuracy_score += 0.4
        accuracy_notes.append("Contains numeric data")
    if has_units:
        accuracy_score += 0.3
        accuracy_notes.append("Includes measurement units")
    if has_time_refs:
        accuracy_score += 0.3
        accuracy_notes.append("References specific times")
    if not accuracy_notes:
        accuracy_notes.append("Missing concrete data points")

    # --- RELEVANCE (0-1) ---
    # Does the response address the key terms from the question?
    relevance_score = 0.0
    relevance_notes = []

    question_keywords = [w for w in re.findall(r'\b\w{4,}\b', question_lower)
                         if w not in {"should", "what", "when", "which", "would", "could", "about", "that", "this", "from", "with", "have", "your"}]
    if question_keywords:
        matched_keywords = sum(1 for kw in question_keywords if kw in text)
        relevance_score = round(min(1.0, matched_keywords / max(1, len(question_keywords) * 0.5)), 2)

    # Check alignment with expected response themes
    expected_tokens = [w for w in re.findall(r'\b\w{4,}\b', expected_lower)]
    if expected_tokens:
        matched_expected = sum(1 for t in expected_tokens[:8] if t in text)
        expected_alignment = min(1.0, matched_expected / max(1, min(len(expected_tokens), 8) * 0.4))
        relevance_score = round(min(1.0, (relevance_score + expected_alignment) / 2), 2)

    if relevance_score >= 0.7:
        relevance_notes.append("Strongly addresses the question")
    elif relevance_score >= 0.4:
        relevance_notes.append("Partially addresses the question")
    else:
        relevance_notes.append("Weak alignment with the question")

    # --- COMPLETENESS (0-1) ---
    # Is the response detailed enough with actionable guidance and reasoning?
    completeness_score = 0.0
    completeness_notes = []

    if len(text) > 300:
        completeness_score += 0.4
        completeness_notes.append("Sufficiently detailed response")
    elif len(text) > 150:
        completeness_score += 0.2
        completeness_notes.append("Moderate detail level")
    else:
        completeness_notes.append("Response may be too brief")

    action_keywords = ["recommend", "should", "schedule", "suggest", "advise", "set", "run", "charge", "avoid"]
    if any(k in text for k in action_keywords):
        completeness_score += 0.3
        completeness_notes.append("Contains actionable guidance")
    else:
        completeness_notes.append("Missing actionable guidance")

    reasoning_keywords = ["because", "since", "reason", "due to", "this means", "therefore", "result"]
    if any(k in text for k in reasoning_keywords):
        completeness_score += 0.3
        completeness_notes.append("Includes reasoning/explanation")
    else:
        completeness_notes.append("Missing reasoning")

    # --- USEFULNESS (0-1) ---
    # Does the response provide practical, usable recommendations?
    usefulness_score = 0.0
    usefulness_notes = []

    time_patterns = bool(re.search(r'\d{1,2}\s*(am|pm|:\d{2})', text))
    if time_patterns:
        usefulness_score += 0.35
        usefulness_notes.append("Provides specific time schedules")
    else:
        usefulness_notes.append("Missing specific time recommendations")

    cost_keywords = ["save", "cost", "saving", "cheaper", "expensive", "price", "$", "bill"]
    if any(k in text for k in cost_keywords):
        usefulness_score += 0.35
        usefulness_notes.append("Includes cost/savings information")
    else:
        usefulness_notes.append("Missing cost/savings context")

    practical_keywords = ["tip", "step", "first", "then", "option", "alternative", "instead"]
    if any(k in text for k in practical_keywords):
        usefulness_score += 0.3
        usefulness_notes.append("Offers practical steps or alternatives")
    else:
        usefulness_notes.append("Could offer more practical alternatives")

    overall_score = round((accuracy_score + relevance_score + completeness_score + usefulness_score) / 4, 2)

    return {
        "accuracy": round(accuracy_score, 2),
        "relevance": round(relevance_score, 2),
        "completeness": round(completeness_score, 2),
        "usefulness": round(usefulness_score, 2),
        "overall": overall_score,
        "passed": overall_score >= 0.5,
        "notes": {
            "accuracy": accuracy_notes,
            "relevance": relevance_notes,
            "completeness": completeness_notes,
            "usefulness": usefulness_notes,
        },
    }

In [20]:
def evaluate_tool_usage(messages, expected_tools):
    """Evaluate if the right tools were used."""
    used_tools = _extract_tool_names(messages)
    used_set = set(used_tools)
    expected_set = set(expected_tools or [])

    matched = sorted(list(expected_set.intersection(used_set)))
    missing = sorted(list(expected_set - used_set))
    extra = sorted(list(used_set - expected_set))

    precision = len(matched) / max(1, len(used_set))
    recall = len(matched) / max(1, len(expected_set))
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

    return {
        "used_tools": used_tools,
        "expected_tools": list(expected_set),
        "matched": matched,
        "missing": missing,
        "extra": extra,
        "precision": round(precision, 2),
        "recall": round(recall, 2),
        "f1": round(f1, 2),
        "passed": recall >= 0.7,
    }

In [22]:
def generate_evaluation_report():
    """Generate evaluation metrics across all test_results with overall scores."""
    report_rows = []
    all_tools_used = set()
    all_tools_expected = set()

    for result in test_results:
        if isinstance(result.get("response"), dict):
            messages = result["response"].get("messages", [])
            final_response = messages[-1].content if messages else ""
        else:
            messages = []
            final_response = str(result.get("response", ""))

        response_eval = evaluate_response(
            question=result["question"],
            final_response=final_response,
            expected_response=result["expected_response"],
        )
        tool_eval = evaluate_tool_usage(messages, result["expected_tools"])

        all_tools_used.update(tool_eval["used_tools"])
        all_tools_expected.update(tool_eval["expected_tools"])

        report_rows.append({
            "test_id": result["test_id"],
            "accuracy": response_eval["accuracy"],
            "relevance": response_eval["relevance"],
            "completeness": response_eval["completeness"],
            "usefulness": response_eval["usefulness"],
            "overall": response_eval["overall"],
            "response_passed": response_eval["passed"],
            "tool_f1": tool_eval["f1"],
            "tool_recall": tool_eval["recall"],
            "tool_passed": tool_eval["passed"],
            "used_tools": tool_eval["used_tools"],
            "missing_tools": tool_eval["missing"],
        })

    total = len(report_rows)

    # Overall scores across all tests
    avg_accuracy = round(sum(r["accuracy"] for r in report_rows) / max(1, total), 2)
    avg_relevance = round(sum(r["relevance"] for r in report_rows) / max(1, total), 2)
    avg_completeness = round(sum(r["completeness"] for r in report_rows) / max(1, total), 2)
    avg_usefulness = round(sum(r["usefulness"] for r in report_rows) / max(1, total), 2)
    avg_overall = round(sum(r["overall"] for r in report_rows) / max(1, total), 2)
    avg_tool_f1 = round(sum(r["tool_f1"] for r in report_rows) / max(1, total), 2)
    pass_rate = round(sum(1 for r in report_rows if r["response_passed"] and r["tool_passed"]) / max(1, total), 2)

    # Tool coverage
    tool_coverage = sorted(all_tools_used)
    missing_from_any = sorted(all_tools_expected - all_tools_used)

    summary = {
        "total_tests": total,
        "overall_scores": {
            "accuracy": avg_accuracy,
            "relevance": avg_relevance,
            "completeness": avg_completeness,
            "usefulness": avg_usefulness,
            "combined_overall": avg_overall,
        },
        "tool_metrics": {
            "avg_tool_f1": avg_tool_f1,
            "tools_used": tool_coverage,
            "tools_never_called": missing_from_any,
        },
        "pass_rate": pass_rate,
        "failed_tests": [r["test_id"] for r in report_rows if not (r["response_passed"] and r["tool_passed"])],
        "details": report_rows,
    }

    return summary

In [23]:
# Generate and display the evaluation report
evaluation_report = generate_evaluation_report()

# --- Overall Combined Scores ---
print("=" * 60)
print("         OVERALL EVALUATION SCORES (All Test Cases)")
print("=" * 60)
scores = evaluation_report["overall_scores"]
print(f"  Accuracy:      {scores['accuracy']:.2f} / 1.00")
print(f"  Relevance:     {scores['relevance']:.2f} / 1.00")
print(f"  Completeness:  {scores['completeness']:.2f} / 1.00")
print(f"  Usefulness:    {scores['usefulness']:.2f} / 1.00")
print(f"  ---")
print(f"  Combined Overall Score:  {scores['combined_overall']:.2f} / 1.00")
print(f"  Overall Pass Rate:       {evaluation_report['pass_rate']:.0%}")

# --- Tool Coverage ---
tools = evaluation_report["tool_metrics"]
print(f"\n  Tool F1 (avg):  {tools['avg_tool_f1']:.2f}")
print(f"  Tools Used:     {', '.join(tools['tools_used'])}")
if tools["tools_never_called"]:
    print(f"  Tools Missing:  {', '.join(tools['tools_never_called'])}")
else:
    print(f"  Tools Missing:  None — all expected tools were called")

# --- Per-Test Breakdown ---
print("\n" + "=" * 60)
print("         PER-TEST BREAKDOWN")
print("=" * 60)
print(f"{'Test ID':<18} {'Acc':>5} {'Rel':>5} {'Comp':>5} {'Use':>5} {'Overall':>8} {'Tools':>6} {'Status':>7}")
print("-" * 60)
for d in evaluation_report["details"]:
    status = "PASS" if d["response_passed"] and d["tool_passed"] else "FAIL"
    print(f"{d['test_id']:<18} {d['accuracy']:>5.2f} {d['relevance']:>5.2f} {d['completeness']:>5.2f} {d['usefulness']:>5.2f} {d['overall']:>8.2f} {d['tool_f1']:>6.2f} {status:>7}")

if evaluation_report["failed_tests"]:
    print(f"\nFailed: {', '.join(evaluation_report['failed_tests'])}")

# --- Strengths ---
print("\n" + "=" * 60)
print("         STRENGTHS")
print("=" * 60)
strengths = []
if scores["accuracy"] >= 0.7:
    strengths.append("High accuracy: responses consistently include concrete data (numbers, units, specific times).")
if scores["relevance"] >= 0.7:
    strengths.append("Strong relevance: responses directly address user questions and align with expected outcomes.")
if scores["completeness"] >= 0.6:
    strengths.append("Good completeness: responses are detailed with actionable guidance and reasoning.")
if scores["usefulness"] >= 0.6:
    strengths.append("Practical usefulness: responses include specific schedules and cost/savings estimates.")
if tools["avg_tool_f1"] >= 0.7:
    strengths.append("Effective tool usage: the agent selects appropriate tools for each query with high F1.")
if not tools["tools_never_called"]:
    strengths.append("Full tool coverage: all available tools are exercised across the test suite.")
if not strengths:
    strengths.append("The agent produces functional responses for all test cases without errors.")
for s in strengths:
    print(f"  + {s}")

# --- Weaknesses ---
print("\n" + "=" * 60)
print("         WEAKNESSES")
print("=" * 60)
weaknesses = []
if scores["accuracy"] < 0.7:
    weaknesses.append("Accuracy could improve: some responses lack specific numbers or measurable data points.")
if scores["relevance"] < 0.7:
    weaknesses.append("Relevance gaps: some responses drift from the core question or miss expected themes.")
if scores["completeness"] < 0.6:
    weaknesses.append("Completeness issues: some responses are brief or lack reasoning for recommendations.")
if scores["usefulness"] < 0.6:
    weaknesses.append("Usefulness gaps: some responses miss specific time schedules or cost estimates.")
if tools["avg_tool_f1"] < 0.7:
    weaknesses.append(f"Tool selection imprecise: average F1 is {tools['avg_tool_f1']:.2f}; agent sometimes skips expected tools or calls extra ones.")
if tools["tools_never_called"]:
    weaknesses.append(f"Some tools never called: {', '.join(tools['tools_never_called'])}.")
if not weaknesses:
    weaknesses.append("No significant weaknesses identified at current thresholds.")
for w in weaknesses:
    print(f"  - {w}")

# --- Recommendations for Improvement ---
print("\n" + "=" * 60)
print("         RECOMMENDATIONS FOR IMPROVEMENT")
print("=" * 60)
recommendations = [
    "1. Enhance the system prompt to explicitly instruct the agent to use the savings calculator tool when comparing usage scenarios, so it provides precise dollar estimates rather than rough approximations.",
    "2. Add prompt guidance for the agent to always check weather forecasts alongside electricity prices for thermostat and HVAC questions, since weather conditions directly affect optimal setpoints.",
    "3. Encourage the agent to cross-reference historical usage data with energy-saving tips (RAG) when giving reduction advice, ensuring recommendations are personalized to the user's actual patterns.",
    "4. Consider adding few-shot examples to the system prompt demonstrating ideal tool combinations for common query types (e.g., EV charging should use weather + pricing + savings calculator).",
    "5. Expand the test suite to cover edge cases such as cloudy-day solar queries, multi-day scheduling, and conflicting optimization goals (cost vs. comfort vs. carbon).",
]
for r in recommendations:
    print(f"  {r}")

print("\n" + "=" * 60)

         OVERALL EVALUATION SCORES (All Test Cases)
  Accuracy:      0.95 / 1.00
  Relevance:     0.90 / 1.00
  Completeness:  0.78 / 1.00
  Usefulness:    0.68 / 1.00
  ---
  Combined Overall Score:  0.83 / 1.00
  Overall Pass Rate:       91%

  Tool F1 (avg):  0.95
  Tools Used:     calculate_energy_savings, get_electricity_prices, get_recent_energy_summary, get_weather_forecast, query_energy_usage, query_solar_generation, search_energy_tips
  Tools Missing:  None — all expected tools were called

         PER-TEST BREAKDOWN
Test ID              Acc   Rel  Comp   Use  Overall  Tools  Status
------------------------------------------------------------
ev_charging_peak_avoid  1.00  1.00  0.70  0.70     0.85   0.80    PASS
ev_charging_weekend_home  1.00  0.97  0.70  0.70     0.84   0.67    FAIL
thermostat_heatwave_peak  1.00  0.97  0.70  0.70     0.84   1.00    PASS
thermostat_night_setback  1.00  0.81  0.70  1.00     0.88   1.00    PASS
laundry_offpeak     1.00  0.97  0.70  1.00     0.